# Robust Historical Valuation Dataset Builder

Builds PE, PB, PS, ROE and revenue growth datasets for S&P500 and NIFTY500.

This notebook includes fixes for common issues:
- Reliable S&P500 source (DataHub instead of Wikipedia scraping)
- Retry logic for API failures
- Rate limiting to avoid blocking
- Safe handling of missing financial fields
- Validation checks for downloaded data
- Modular pipeline for ML dataset generation


## Step 0 — Install dependencies

In [3]:
!pip install yfinance pandas tqdm pyarrow --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Imports

In [4]:
import pandas as pd
import yfinance as yf
import time
from tqdm import tqdm


## Step 2 — Load S&P500 universe (stable source)

In [5]:
sp500 = pd.read_csv(
    'https://datahub.io/core/s-and-p-500-companies/r/constituents.csv'
)

tickers_sp500 = sp500['Symbol'].tolist()

print('Total S&P500 tickers:', len(tickers_sp500))
tickers_sp500[:10]

Total S&P500 tickers: 503


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']

## Step 3 — Load NIFTY500 tickers
Place `nifty500.csv` in the same folder with column `Symbol`.

In [6]:
try:
    nifty = pd.read_csv('nifty500.csv')
    tickers_nifty = [s + '.NS' for s in nifty['Symbol'].tolist()]
except:
    tickers_nifty = []
    print('NIFTY500 file not found, skipping...')

print('NIFTY tickers:', len(tickers_nifty))

NIFTY500 file not found, skipping...
NIFTY tickers: 0


## Step 4 — Helper: retry wrapper

In [7]:
def retry(func, retries=3):
    for i in range(retries):
        try:
            return func()
        except Exception as e:
            print('Retry', i+1, 'error:', e)
            time.sleep(2)
    return None

## Step 5 — Download price history

In [14]:
def get_price(ticker):

    df = yf.download(
        ticker,
        start="2010-01-01",
        progress=False,
        auto_adjust=False   # FIX
    )

    df = df.reset_index()
    df["ticker"] = ticker

    return df

## Step 6 — Download fundamentals

In [9]:
def get_fundamentals(ticker):

    def fetch():

        t = yf.Ticker(ticker)

        income = t.quarterly_income_stmt.T
        balance = t.quarterly_balance_sheet.T

        if income.empty or balance.empty:
            return None

        df = pd.DataFrame()

        df['eps'] = income.get('Diluted EPS')
        df['revenue'] = income.get('Total Revenue')
        df['net_income'] = income.get('Net Income')

        df['equity'] = balance.get('Total Stockholder Equity')

        df['date'] = df.index
        df['ticker'] = ticker

        return df.reset_index(drop=True)

    return retry(fetch)

## Step 7 — Merge price + fundamentals

In [10]:
def merge_data(price, fundamentals):

    fundamentals['date'] = pd.to_datetime(fundamentals['date'])

    merged = pd.merge_asof(
        price.sort_values('Date'),
        fundamentals.sort_values('date'),
        left_on='Date',
        right_on='date',
        direction='backward'
    )

    return merged

## Step 8 — Compute valuation factors

In [11]:
def compute_factors(df):

    df['PE'] = df['Close'] / df['eps']

    df['PB'] = df['Close'] / (df['equity'] / 1e9)

    df['PS'] = df['Close'] / (df['revenue'] / 1e9)

    df['ROE'] = df['net_income'] / df['equity']

    df['revenue_growth'] = df['revenue'].pct_change(4)

    return df

## Step 9 — Build dataset (with validation)

In [15]:
all_tickers = tickers_sp500 + tickers_nifty

dataset = []

for ticker in tqdm(all_tickers):

    price = get_price(ticker)
    fundamentals = get_fundamentals(ticker)

    if price is None or fundamentals is None:
        continue

    try:
        merged = merge_data(price, fundamentals)
        factors = compute_factors(merged)
        dataset.append(factors)

    except Exception as e:
        print('Processing error:', ticker, e)

    time.sleep(0.5)

  0%|                                                                                                 | 0/503 [00:00<?, ?it/s]

Processing error: MMM Not allowed to merge between different levels. (2 levels on the left, 1 on the right)


  0%|▏                                                                                        | 1/503 [00:01<15:21,  1.84s/it]

Processing error: AOS Not allowed to merge between different levels. (2 levels on the left, 1 on the right)


  0%|▎                                                                                        | 2/503 [00:03<15:07,  1.81s/it]

...

100%|██████████████████████████████████████████████████████████████████████████████████████▋| 501/503 [16:06<00:06,  3.04s/it]

Processing error: ZBH Not allowed to merge between different levels. (2 levels on the left, 1 on the right)


100%|██████████████████████████████████████████████████████████████████████████████████████▊| 502/503 [16:08<00:02,  2.74s/it]

Processing error: ZTS Not allowed to merge between different levels. (2 levels on the left, 1 on the right)


100%|███████████████████████████████████████████████████████████████████████████████████████| 503/503 [16:10<00:00,  1.93s/it]


## Step 10 — Combine results

In [ ]:
final_df = pd.concat(dataset)

print('Final dataset size:', final_df.shape)

final_df.head()

## Step 11 — Save dataset

In [ ]:
final_df.to_parquet('valuation_dataset.parquet')
final_df.to_csv('valuation_dataset.csv', index=False)

print('Dataset saved successfully')